# Step 1: Data Preparation

In this notebook, we:
1. Read the `reviews83325.csv` and `Tripadvisor.csv` databases.
2. Filter reviews to keep only English reviews to reduce vocabulary entropy.
3. Group variable number of reviews per place.
4. Use Term Frequency-Inverse Document Frequency (TF-IDF) to extract the best 100 words summarizing a place, acting as our normalization mechanism.

In [4]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

### 1. Load Datasets & EDA

In [5]:
df_reviews = pd.read_csv('../data/reviews83325.csv')
df_places = pd.read_csv('../data/Tripadvisor.csv')

print(f"Total reviews: {len(df_reviews)}")
print(f"Total places: {len(df_places)}")

Total reviews: 340385
Total places: 3761


### 2. Filter English Reviews

In [6]:
df_reviews_en = df_reviews[df_reviews['langue'] == 'en'].copy()
print(f"English reviews: {len(df_reviews_en)}")

English reviews: 153071


### 3. Group Reviews by Place

In [7]:
df_grouped = df_reviews_en.groupby('idplace')['review'].apply(lambda x: ' '.join(x.dropna().astype(str))).reset_index()
print(f"Places with English reviews: {len(df_grouped)}")

Places with English reviews: 1835


### 4. TF-IDF Normalization (Top 100 Words)

In [8]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=100)

def extract_top_100_tfidf(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ""
    try:
        X = vectorizer.fit_transform([text])
        scores = zip(vectorizer.get_feature_names_out(), np.asarray(X.sum(axis=0)).ravel())
        sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)
        return " ".join([word for word, score in sorted_scores])
    except ValueError:
        return text

df_grouped['top_100_words'] = df_grouped['review'].apply(extract_top_100_tfidf)
df_grouped.head()

,idplace,review,top_100_words
0,188467,Personally I think it is the most beautiful sq...,place square paris park beautiful des vosges m...
1,188468,My old college friend and I booked this beauti...,street shopping apartment marais day des great...
2,188470,"Being winter and all, not a lot going on, howe...",shops paul village st place little paris area ...
3,188471,To call Au Passe Partout a shop is a serious u...,history owner unique absolutely au bought chea...
4,188472,Very old historical place. I attended to exper...,paris place art billettes church cloister des ...


### 5. Format and Export the Prepared Dataset

In [9]:
# Save the prepared data to a new CSV file to inject directly into the ML models
df_final = df_grouped[['idplace', 'top_100_words', 'review']]
df_final.to_csv('prepared_reviews.csv', index=False)
print("Cleaned reviews dataset saved as 'prepared_reviews.csv'")

Cleaned reviews dataset saved as 'prepared_reviews.csv'
